In [0]:
from pyspark.sql.functions import col, lit, to_timestamp, current_timestamp

In [0]:
dbutils.widgets.text("source_system", "")
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("landing_timestamp", "")
dbutils.widgets.text("source_file_format", "")

In [0]:
source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_file_format = dbutils.widgets.get("source_file_format")

In [0]:
landing_path = f"/Volumes/adb_caresync/landing/landing_volume/{source_system}/{table_name}/{landing_timestamp}/"

try:
    landing_files = dbutils.fs.ls(landing_path)
except Exception:
    landing_files = []

if not landing_files:
    print(f"No new file received at {landing_path}. Exiting notebook.")
    dbutils.notebook.exit("No new file received")

In [0]:
file_format = source_file_format.lower().strip()
if not file_format:  # infer from landing file extension when ADF passes an empty value
    file_format = landing_files[0].name.rsplit(".", 1)[-1].lower()

if file_format == "csv":  # Check the value of source_file_format before comparison
    landing_df = spark.read.format("csv")\
                           .option("header", True)\
                           .option("inferSchema", True)\
                           .load(landing_path)

elif file_format == "parquet":  # Check the value of source_file_format before comparison
    landing_df = spark.read.format("parquet") \
               .load(landing_path)

else:
    raise ValueError(f"Unsupported source_file_format '{source_file_format}' for {landing_path}")

# display(landing_df)               

In [0]:
# adding two columns in the landng_df

landing_df = landing_df.withColumn(
    "landing_timestamp", lit(landing_timestamp)
).withColumn(
    "insert_timestamp", lit(current_timestamp())
)

In [0]:
landing_df.write.mode("append")\
                .saveAsTable(f"adb_caresync.bronze.{table_name}")